# Taller 3: Clasificación de datos utilizando imágenes

In [29]:
import glob
import cv2
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input, BatchNormalization, GlobalAveragePooling2D, Rescaling, RandomZoom, RandomRotation, RandomFlip, Rescaling
import pandas as pd
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import os
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.data import AUTOTUNE
from tensorflow.keras.losses import SparseCategoricalCrossentropy
import seaborn as sns
import matplotlib.pyplot as plt

## Procesado de imagenes

In [30]:
imagenes_train = []
labels_train = []

for label, folder in enumerate(glob.glob("train/*")):
    for img_path in glob.glob(f"{folder}/*.png"):
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        img = cv2.resize(img, (128, 128)).flatten()
        imagenes_train.append(img)
        labels_train.append(label)
    
imagenes_train = np.array(imagenes_train)
labels_train = np.array(labels_train)

## Regresión logistica

In [ ]:
# Cargar y preprocesar las imágenes de entrenamiento
scaler = StandardScaler()
imagenes_train_escaladas = scaler.fit_transform(imagenes_train)

# Entrenar el modelo de regresión logística
logreg = LogisticRegression(max_iter=10000)
logreg.fit(imagenes_train_escaladas, labels_train)

# Evaluar el modelo
predicciones_train = logreg.predict(imagenes_train)
puntaje_train = accuracy_score(labels_train, predicciones_train)
print("Accuracy en entrenamiento:", puntaje_train)

# Obtener la matriz de confusión
matriz_confusion = confusion_matrix(labels_train, predicciones_train)

# Obtener las etiquetas de las clases
clases_labels = [folder.split("\\")[-1] for folder in sorted(glob.glob("train/*"))]

# Crear un mapa de calor de la matriz de confusión
plt.figure(figsize=(8, 6))
sns.heatmap(matriz_confusion, annot=True, fmt="d", cmap="Blues", xticklabels=clases_labels, yticklabels=clases_labels)

# Añadir título y etiquetas
plt.title('Matriz de Confusión - Regresión Logística')
plt.xlabel('Predicción')
plt.ylabel('Valor Real')
plt.show()

# Procesar imágenes de prueba (si es necesario para el análisis posterior)
imagenes_test = []
archivos_test = []

for img_path in glob.glob("test/*.png"):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (128, 128)).flatten()
    imagenes_test.append(img)
    archivos_test.append(img_path.split("\\")[-1])

imagenes_test = np.array(imagenes_test)
imagenes_test_escalada = scaler.transform(imagenes_test)

# Predicciones sobre el conjunto de prueba
predicciones_test = logreg.predict(imagenes_test_escalada)
labels_predictions_test = [clases_labels[pred] for pred in predicciones_test]

# Guardar los resultados en un archivo CSV
resultados = pd.DataFrame({
    "file": archivos_test,
    "label": labels_predictions_test
})
resultados.to_csv("resultado_log.csv", index=False)

## Red covolucional

In [ ]:
batch_size = 32
img_height = 180
img_width = 180
dir = "train"

train_dataset = image_dataset_from_directory(
  dir,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

validacion_dataset = image_dataset_from_directory(
  dir,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

image_class_labels = train_dataset.class_names

Found 5406 files belonging to 16 classes.
Using 4325 files for training.
Found 5406 files belonging to 16 classes.
Using 1081 files for validation.


In [ ]:
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validacion_dataset = validacion_dataset.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
num_classes = len(image_class_labels)

data_augmentation = Sequential([
  RandomFlip("horizontal", input_shape=(img_height, img_width, 3)),
  RandomRotation(0.1),
  RandomZoom(0.1),
])

model = Sequential([
  data_augmentation,
  Rescaling(1./255, input_shape=(img_height, img_width, 3)),
  Conv2D(16, 3, padding='same', activation='relu'),
  MaxPooling2D(),
  Conv2D(32, 3, padding='same', activation='relu'),
  MaxPooling2D(),
  Conv2D(64, 3, padding='same', activation='relu'),
  MaxPooling2D(),
  Flatten(),
  Dense(128, activation='relu'),
  Dense(num_classes, activation='softmax')
])

c:\Users\La maquina\Desktop\Taller1Estadistica\.venv\Lib\site-packages\keras\src\layers\preprocessing\tf_data_layer.py:19: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
model.compile(optimizer='adam',loss=SparseCategoricalCrossentropy(from_logits=True),metrics=['accuracy'])

In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
history = model.fit(
  train_dataset,
  validation_data=validacion_dataset,
  epochs=50,
  callbacks=[early_stopping]
)

validacion_predicciones = model.predict(validacion_dataset)
validacion_predicciones_clases = np.argmax(validacion_predicciones, axis=1)
val_labels = np.concatenate([y for x, y in validacion_dataset], axis=0)

print("Matriz de confusión Regresion Logistica:\n", confusion_matrix(val_labels, validacion_predicciones_clases))

Epoch 1/50


c:\Users\La maquina\Desktop\Taller1Estadistica\.venv\Lib\site-packages\keras\src\backend\tensorflow\nn.py:635: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


136/136 ━━━━━━━━━━━━━━━━━━━━ 24s 165ms/step - accuracy: 0.1685 - loss: 2.4940 - val_accuracy: 0.3867 - val_loss: 1.7508
Epoch 2/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 23s 168ms/step - accuracy: 0.4396 - loss: 1.5922 - val_accuracy: 0.5254 - val_loss: 1.3896
Epoch 3/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 22s 162ms/step - accuracy: 0.5164 - loss: 1.3807 - val_accuracy: 0.5458 - val_loss: 1.2977
Epoch 4/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 22s 161ms/step - accuracy: 0.6082 - loss: 1.1292 - val_accuracy: 0.5994 - val_loss: 1.2599
Epoch 5/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 22s 165ms/step - accuracy: 0.6475 - loss: 1.0035 - val_accuracy: 0.6698 - val_loss: 0.9587
Epoch 6/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 21s 155ms/step - accuracy: 0.7015 - loss: 0.8669 - val_accuracy: 0.7049 - val_loss: 1.0211
Epoch 7/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 21s 156ms/step - accuracy: 0.7311 - loss: 0.7770 - val_accuracy: 0.6744 - val_loss: 1.0255
Epoch 8/50
136/136 ━━━━━━━━━━━━━━━━━━━━ 21s 156ms/step - accuracy: 0.7744 - loss: 0.6736 - val

In [ ]:
dir_test = "test"
ruta_img_test = glob.glob(os.path.join(dir_test, "*.png"))

def load_and_preprocess_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [img_height, img_width])
    img = tf.expand_dims(img, 0)
    return img

predicciones = []
nombres_imagenes = []

for img_path in ruta_img_test:
        img = load_and_preprocess_image(img_path)
        preds = model.predict(img)
        predicciones_class = np.argmax(preds, axis=1)
        predicted_class_name = image_class_labels[predicciones_class[0]]
        predicciones.append(predicted_class_name)
        nombres_imagenes.append(img_path)

resultados = pd.DataFrame({
        "file": [os.path.basename(name) for name in nombres_imagenes],
        "label": predicciones
})

resultados.to_csv("resultado_cnn.csv", index=False)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━